# Exploration notebook

Template for cross-experiment analysis. Edit freely — gitignored after initial commit.
Promote any reusable pattern to `analysis/plot.py`.

In [ ]:
import sys; sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import analysis.plot as P

from analysis.load import load_results, filter_results, to_dataframe, load_scene
from analysis.plot import (metric_vs_param, convergence_curves, compare_algorithms,
                            sanity_check, diff_error_map, sam_map,
                            show_pixel, show_region, spectral_profile,_to_rgb)

# ── Global switches ────────────────────────────────────────────
P.SAVE_FIGURES = False   # True  → save PNGs to OUTPUT_DIR
                         # False → display inline
STUDY_DIR  = '../results/robustness_study_20260320_191432'   # ← point here
OUTPUT_DIR = '../figs/explore'
# ──────────────────────────────────────────────────────────────

PARAMS  = ['algorithm', 'lmbda', 'lmbda_m', 'p', 'q', 'r',
           'scale', 'noise_level', 'sigma_blur', 'max_iter', 'max_iter_cp', 'threshold_softness']
METRICS = ['PSNR_mean', 'SSIM_mean', 'SAM_mean', 'RNMSE_mean', 'CC_mean']

results = load_results(STUDY_DIR)
df = to_dataframe(results)
print(f'{len(df)} experiments | {df["algorithm"].value_counts().to_dict()}')

## 1. Coverage — what has been run?

In [ ]:
# Full table, sorted by PSNR
cols = [c for c in PARAMS if c in df.columns] + [c for c in METRICS if c in df.columns]
df[cols].sort_values('PSNR_mean', ascending=False)

In [ ]:
# Parameter space coverage — adjust axes to what you care about
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x='lmbda', y='noise_level',
                hue='algorithm', size='PSNR_mean', sizes=(40, 200), ax=ax)
ax.set_xscale('log')
ax.set_title('Explored parameter space')
plt.tight_layout()

## 2. Define your slice

Set `FILTERS` once. All cells below use `subset` / `subset_df`.
Comment out keys to leave that dimension free.

In [ ]:
# ── Edit here ─────────────────────────────────────────────────
FILTERS = dict(
    # lmbda = 0.001,
    lmbda_m = 5.0,
    # noise_level=37,
    # threshold_softness = 1.0e-5,
    # sigma_blur = 8.0,
)
# ──────────────────────────────────────────────────────────────

subset    = filter_results(results, **FILTERS)
subset_df = to_dataframe(subset)

free = subset_df[[c for c in PARAMS if c in subset_df.columns]]
free_params = free.columns[free.nunique() > 1].tolist()
fixed_params = free.columns[free.nunique() == 1].to_list()

counts    = subset_df.groupby(free_params)['PSNR_mean'].transform('count')                                                                                                  
is_dup    = counts > 1                                                                                                                                                      
                                                                                                                                                                            
# inspect before acting                                                                                                                                                     
# print(subset_df[is_dup][free_params + ['PSNR_mean', 'zarr_path']].sort_values(free_params))                                                                                 
                                                                                                                                                                            
# then drop the worse one only within duplicate groups                                                                                                                      
idx_drop  = subset_df[is_dup].groupby(free_params)['PSNR_mean'].idxmin()                                                                                                    
subset_df = subset_df.drop(index=idx_drop).reset_index(drop=True)                                                                                                           
subset    = [r for i, r in enumerate(subset) if i not in idx_drop] 


# print(duplicates[free_params].sort_values(free_params)) 


print(f'{len(subset)} experiments match')
print(f'Fixed parameters: {fixed_params}')
print(subset_df[[c for c in fixed_params]].iloc[0])
print(f'Free parameters: {free_params}')
subset_df[[c for c in free_params + [m for m in METRICS if m in subset_df.columns]]]


# duplicates = subset_df.groupby(free_params).filter(lambda g: len(g) > 1)



In [ ]:

target_params = 'lmbda'
target_params_value = [0.01, 0.001]

other_params = [p for p in free_params if p!= target_params]

complete = (subset_df.groupby(other_params).filter(lambda g: set(target_params_value).issubset(g['lmbda'])))

complete[other_params + ['lmbda']]



## 3. Sanity check — one experiment

Set `i` to navigate through experiments in the current slice.

In [ ]:
idxs = [1,6]  # ← 0 … len(subset)-1


for i in idxs:
    r = subset[i]   
    print({k: r.get(k) for k in free_params + ['algorithm']})

    scene, rgb_index = load_scene(r)
    sanity_check(r, scene, output_dir=OUTPUT_DIR, rgb_indices=rgb_index)

    

## 4. Spatial comparison — CTV vs GradAlign

Pick matched experiments (same params, different algorithm) and compare spatially.

In [ ]:
# Fix everything except algorithm
# ── Edit here ─────────────────────────────────────────────────
FILTERS = dict(
    lmbda = 0.01,
    lmbda_m = 5.0,
    noise_level=37,
    threshold_softness = 1.0e-5,
    sigma_blur = 1.0,
)

image_idx = 0


fixed = {k: v for k, v in FILTERS.items()}
r_ctv = filter_results(results, algorithm='CTV',       **fixed)[0]
r_ga  = filter_results(results, algorithm='GradAlign', **fixed)[0]

scene, rgb_index = load_scene(r_ctv, image_idx=image_idx)   # same degradation → same scene for both

diff_error_map(r_ctv['reconstructed'][image_idx], r_ga['reconstructed'][image_idx],
               scene['gt'], scene['ym'],
               label_a='CTV', label_b='GradAlign', output_dir=OUTPUT_DIR)

In [ ]:
# SAM maps
sam_ctv = sam_map(r_ctv['reconstructed'][image_idx], scene['gt'],
                  title='SAM — CTV', output_dir=OUTPUT_DIR)
sam_ga  = sam_map(r_ga['reconstructed'][image_idx],  scene['gt'],
                  title='SAM — GradAlign', output_dir=OUTPUT_DIR)
print(f'Mean SAM  CTV: {sam_ctv.mean():.3f}°   GradAlign: {sam_ga.mean():.3f}°')

In [ ]:
%matplotlib widget                                                                                                                                                          
from matplotlib.widgets import RectangleSelector                                                                                                                            
                                                                                                                                                                            
# ── reconstructions to compare ─────────────────────────────────                                                                                                           
RECONS = {
    'CTV':       r_ctv['reconstructed'][image_idx],                                                                                                                                 
    'GradAlign': r_ga['reconstructed'][image_idx],                                                                                                                                  
}                                                                                                                                                                           
# ──────────────────────────────────────────────────────────────                                                                                                            
                                                                                                                                                                            
fig, (ax_img, ax_spec) = plt.subplots(1, 2, figsize=(14, 5))                                                                                                                
ax_img.imshow(_to_rgb(scene['gt'], rgb_index))                                                                                                                              
ax_img.set_title('Draw a region')                                                                                                                                           
                
def _stats(arr, ymin, ymax, xmin, xmax):                                                                                                                                    
    patch = arr[:, ymin:ymax, xmin:xmax].reshape(arr.shape[0], -1)
    return patch.mean(axis=1), patch.std(axis=1)                                                                                                                            
                                                                                                                                                                            
def on_select(eclick, erelease):
    xmin = int(min(eclick.xdata, erelease.xdata))                                                                                                                           
    xmax = int(max(eclick.xdata, erelease.xdata))
    ymin = int(min(eclick.ydata, erelease.ydata))                                                                                                                           
    ymax = int(max(eclick.ydata, erelease.ydata))
    if xmax == xmin or ymax == ymin:                                                                                                                                        
        return                                                                                                                                                              

    ax_spec.cla()                                                                                                                                                           
    bands = np.arange(scene['gt'].shape[0])
                                                                                                                                                                            
    mean, std = _stats(scene['gt'], ymin, ymax, xmin, xmax)                                                                                                                 
    ax_spec.plot(bands, mean, 'k-', linewidth=2, label='GT')                                                                                                                
    ax_spec.fill_between(bands, mean - std, mean + std, alpha=0.15, color='k')                                                                                              
                                                                                                                                                                            
    for name, recon in RECONS.items():
        mean, std = _stats(recon, ymin, ymax, xmin, xmax)                                                                                                                   
        line, = ax_spec.plot(bands, mean, '--', linewidth=1.5, label=name)
        ax_spec.fill_between(bands, mean - std, mean + std, alpha=0.15, color=line.get_color())                                                                             
                                                                                                                                                                            
    ax_spec.set_title(f'y=[{ymin}:{ymax}]  x=[{xmin}:{xmax}]')                                                                                                              
    ax_spec.set_xlabel('Band index'); ax_spec.set_ylabel('Intensity')                                                                                                       
    ax_spec.legend(); ax_spec.grid(True, alpha=0.3)                                                                                                                         
    print(f'region_yx = ({ymin}, {ymax}, {xmin}, {xmax})')   # ← copy to save                                                                                               
    fig.canvas.draw()                                                                                                                                                       
                                                                                                                                                                            
rs = RectangleSelector(ax_img, on_select, useblit=True,                                                                                                                     
                        button=[1], minspanx=5, minspany=5,
                        spancoords='pixels', interactive=True)                                                                                                               
plt.tight_layout() # copy-paste into spectral_profile cell 

In [ ]:
# Define regions — one at a PAN edge, one in a smooth area
# Use show_region to pick coordinates visually first
region_edge   = (100,110, 81, 85)   # ← (ymin, ymax, xmin, xmax)
region_smooth = (56, 60, 62,72)

show_region(scene['gt'], region_edge,   output_dir=OUTPUT_DIR,
            rgb_indices=rgb_index, title='Edge region')
show_region(scene['gt'], region_smooth, output_dir=OUTPUT_DIR,
            rgb_indices=rgb_index, title='Smooth region')

In [ ]:
# Spectral profiles — mean ± 1σ over each region
spectral_profile(region_edge, scene['gt'], output_dir=OUTPUT_DIR,
                 CTV=r_ctv['reconstructed'][image_idx],
                 GradAlign=r_ga['reconstructed'][image_idx])

spectral_profile(region_smooth, scene['gt'], output_dir=OUTPUT_DIR,
                 CTV=r_ctv['reconstructed'][image_idx],
                 GradAlign=r_ga['reconstructed'][image_idx])

## 5. Global metrics and convergence

In [ ]:
compare_algorithms({'CTV': filter_results(subset, algorithm='CTV'),
                    'GradAlign': filter_results(subset, algorithm='GradAlign')},
                   'PSNR', output_dir=OUTPUT_DIR)

In [ ]:
metric_vs_param(subset, param='lmbda', metric_name='PSNR', output_dir=OUTPUT_DIR)

In [ ]:
convergence_curves(subset, group_by='lmbda', facet_by='algorithm',
                   mode='distance', output_dir=OUTPUT_DIR)

## 6. Deep dive — free-form

In [ ]:
# Two-parameter interaction
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(subset_df['lmbda'], subset_df['noise_level'],
                c=subset_df['PSNR_mean'], cmap='viridis', s=80)
plt.colorbar(sc, ax=ax, label='PSNR')
ax.set_xscale('log')
ax.set_xlabel('lmbda'); ax.set_ylabel('noise_level')
ax.set_title('PSNR over (lmbda, noise_level)')
plt.tight_layout()